In [79]:
import numpy as np 
import pandas as pd 

df = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/Divar.csv")
city_classification = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/iran_city_classification.csv")


/tmp/ipykernel_16755/1591124377.py:4: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/Divar.csv")


In [80]:
df.columns.tolist()
df['cat3_slug'].value_counts()

cat3_slug
apartment-sell                        303385
apartment-rent                        211880
plot-old                              133570
house-villa-sell                      121753
house-villa-rent                       64678
shop-rent                              45993
shop-sell                              21855
office-rent                            21418
suite-apartment                        16465
presell                                15781
villa                                  12899
industry-agriculture-business-sell     11851
industry-agriculture-business-rent      9155
office-sell                             5155
partnership                             3622
workspace                                539
Name: count, dtype: int64

In [81]:
num_cols = [
    'building_size', 'floor', 
    'rooms_count', 'total_floors_count', 
    'unit_per_floor', 'construction_year'
]

In [82]:
df[num_cols].head()
#unit per floor : تعداد واحدهای هر طبقه 

,building_size,floor,rooms_count,total_floors_count,unit_per_floor,construction_year
0,500.0,NaN,سه,NaN,NaN,NaN
1,60.0,3,یک,NaN,NaN,۱۳۸۴
2,132.0,3,سه,NaN,NaN,۱۴۰۱
3,90.0,4,یک,NaN,NaN,۱۴۰۰
4,115.0,4,دو,6,NaN,۱۴۰۳


In [83]:
df[num_cols].dtypes

building_size         float64
floor                  object
rooms_count               str
total_floors_count     object
unit_per_floor            str
construction_year         str
dtype: object

In [84]:
trans_table = str.maketrans('۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789')

df['construction_year'] = df['construction_year'].astype(str).str.translate(trans_table)

# با این کار "قبل از 1370" تبدیل به "1370" می‌شود
df['construction_year'] = df['construction_year'].str.extract(r'(\d+)')[0]

# ۴. تبدیل نهایی به نوع عددی (با پوشش دادن مقادیر مفقود احتمالی)
df['construction_year'] = pd.to_numeric(df['construction_year'], errors='coerce')

# ۵. در صورت نیاز به نوع Int64 (پشتیبانی از Int همراه با NaN)
df['construction_year'] = df['construction_year'].astype('Int64')

df["construction_year"].value_counts()

# نمایش تمام مقادیر یکتا همراه با تعداد تکرار
print(df['construction_year'].value_counts(dropna=False))

construction_year
<NA>    184172
1403    116260
1390     59139
1402     58424
1400     53674
1395     53029
1398     38207
1397     36326
1396     35487
1401     35328
1385     34065
1399     29594
1393     29094
1392     26130
1394     26110
1388     24268
1380     23480
1370     20637
1389     16755
1391     16316
1387     14136
1386     13468
1383      9894
1384      8494
1375      7247
1382      6965
1371      5531
1381      3590
1378      3025
1379      2415
1377      2117
1372      1914
1373      1827
1376      1593
1374      1289
Name: count, dtype: Int64


In [85]:
print(df["building_size"].isnull().sum() / len(df["building_size"]) * 100 )
df["building_size"].value_counts()


1.9606


building_size
100.0       49516
80.0        33854
120.0       32562
90.0        32420
75.0        28305
            ...  
1501.0          1
16208.0         1
1161.0          1
585688.0        1
1548.0          1
Name: count, Length: 3572, dtype: int64

In [86]:
# rooms count 
rooms_map = {
    'بدون اتاق': 0,
    'یک': 1,
    'دو': 2,
    'سه': 3,
    'چهار': 4,
    'پنج یا بیشتر': 5
}

df['rooms_count'] = df['rooms_count'].map(rooms_map)

# در صورت وجود مقدار مفقود (NaN)، تبدیل به نوع عددی Int64 پشتیبانی‌کننده از NaN
df['rooms_count'] = df['rooms_count'].astype('Int64')

In [87]:
#floor 
df['floor'] = (
    df['floor']
    .astype(str)
    .str.strip()
    .str.replace('30+', '30', regex=False)  # تبدیل +30 به 30
    .str.replace(r'\.0$', '', regex=True)  # تبدیل 2.0 به 2
)

df['floor'] = pd.to_numeric(df['floor'], errors='coerce').astype('Int64')

In [88]:
#totoal_floor-count 

df['total_floors_count'] = (
    df['total_floors_count']
    .astype(str)
    .str.strip()
    .str.replace('unselect', '', regex=False)
    .str.replace('30+', '30', regex=False)     
    .str.replace(r'\.0$', '', regex=True)   
)

df['total_floors_count'] = pd.to_numeric(df['total_floors_count'], errors='coerce').astype('Int64')

In [89]:
#unit_per_floor

df['unit_per_floor'] = (
    df['unit_per_floor']
    .astype(str)
    .str.strip()
    .str.replace('more_than_8', '9', regex=False)
    .str.replace('unselect', '', regex=False)
)

df['unit_per_floor'] = pd.to_numeric(df['unit_per_floor'], errors='coerce').astype('Int64')

In [90]:
#land_size
# بیشتر از 81 درصد از داده های این ستون نامشخص هستند
# تصمیم شد که یک به وسیله این یک ویژگی جدید بسازیم 
#میتونیم اینجوری برداشت کنیم در خانه هایی که این ویژگی خالی است یک خانه آپارتمانی  هست و چون زمین متعلق به آپارتمان هست٬ متراژ اون رو وارد نمی کنیم
# ساخت ویژگی باینری و حذف ستون اصلی

df['has_land'] = df['land_size'].notnull().astype('Int64')
df = df.drop(columns=['land_size'])

In [91]:
# ۱. برای املاک مسکونی (residential): متراژ بالای ۱,۰۰۰ متر معمولاً غیرمنطقی است
residential_mask = df['cat2_slug'].str.contains('residential', na=False)
df.loc[residential_mask & ((df['building_size'] < 10) | (df['building_size'] > 1000)), 'building_size'] = np.nan

# ۲. برای املاک تجاری/اداری/خدماتی (commercial): سقف متراژ را روی ۲۰,۰۰۰ متر می‌گذاریم
commercial_mask = df['cat2_slug'].str.contains('commercial', na=False)
df.loc[commercial_mask & ((df['building_size'] < 10) | (df['building_size'] > 20000)), 'building_size'] = np.nan

# ۳. برای سایر دسته‌ها (زمین/کلنگی، اجاره کوتاه مدت و ...): متراژهای پرت کلی (زیر ۱۰ یا بالای ۱۰,۰۰۰ متر) -> NaN
other_mask = ~residential_mask & ~commercial_mask
df.loc[other_mask & ((df['building_size'] < 10) | (df['building_size'] > 10000)), 'building_size'] = np.nan

In [92]:
print("حداقل متراژ:", df['building_size'].min())
print("حداکثر متراژ:", df['building_size'].max())

حداقل متراژ: 10.0
حداکثر متراژ: 20000.0


In [93]:
df["building_size"].shape
df["building_size"].isnull().sum() / len(df["building_size"]) * 100

np.float64(5.0931)

In [ ]:
res_outliers = df[(df['cat2_slug'].str.contains('residential', na=False)) & (df['building_size'] > 1000)]
print("تعداد پرت‌های مسکونی باقی‌مانده:", len(res_outliers))

تعداد پرت‌های مسکونی باقی‌مانده: 0
